# The reader trial — can a language model make the reading a person makes?

**What this measures.** Our agent's organs execute (scene tracker, laws by press, planner, attempt ledger); what none of them does on a game never met is *make the reading* — "crates go into the pen", "the orange one is helping", "that line is a wall I can't pass but the boxes can". This notebook hands a language model the same scene our agent sees (the things the tracker names + the frame as a grid) and asks for the reading in the agent's own relation language (JSON). It records the answer, the latency, and whether it matches the human reading (gold) for 44 public scenes (25 first levels + 19 second levels).

**Inputs (right panel).**
1. Dataset `readerEval` (dataset.json, gold.json) — already attached.
2. The model the leaders use: `Intel-Qwen3.8-Flash-Next-W4A16-AutoRound` (framework Transformers, best public 34.3), on the competition environment's **GPU RTX Pro 6000** (96 GB) — Settings → Accelerator. Settings → Turn on internet, so the install cell can fetch `auto-round`. (On a T4 x2 session this model does not fit; use the Qwen team's `Qwen3` 8b there instead and set `DTYPE = torch.float16`.)

**How to run.** Run the cells one at a time with ▶ (not Save Version). The loading cell must print `loaded on cuda:0`. Expect 20–60 s per scene on T4s, about an hour for all 44. The answers are saved to `/kaggle/working/reader_answers.json` — download it from the Output section when done.

**What to read off the result.** (a) seconds per call — the agent has ~245 s of thinking per game in the 9-hour rerun; (b) the gold match count in the last cell; (c) the readings on scenes with no gold (sp80, sc25, sk48, dc22, g50t, sb26 — the games the core has never won): judge them by eye against the frame.

In [ ]:
# Internet is off on the RTX Pro 6000 in this competition, so auto-round (pure Python) and py-cpuinfo come from the attached
# ar-wheels dataset.  transformers stays as the Kaggle image has it (5.16.1): that version already knows the checkpoint's
# n-gram table is stored as 128 shards (ngram_embedding.shard_N) and concatenates them on the CPU.
import glob, subprocess, sys, os
# must be set before torch initialises CUDA: lets the allocator grow/shrink segments instead of carving one huge block
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
whl = sorted(set(glob.glob('/kaggle/input/**/auto_round-*.whl', recursive=True)))
assert whl, 'attach the ar-wheels dataset (Add Input -> Datasets -> ar-wheels)'
wdir = whl[0].rsplit('/', 1)[0]
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', '--find-links', wdir, 'auto-round', 'py-cpuinfo'],
                   capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
import importlib.metadata as md
tv = md.version('transformers'); print('transformers', tv, '(need >= 5.16)')
assert tuple(int(x) for x in tv.split('.')[:2]) >= (5, 16), 'transformers too old for the sharded n-gram table'
print('auto-round', md.version('auto-round'))
import torch; print('torch', torch.__version__, 'gpu', torch.cuda.get_device_name(0), round(torch.cuda.mem_get_info()[1]/2**30), 'GB')
ram = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30
print('host RAM %.0f GiB' % ram, '-- the n-gram table (bf16, ~95 GiB) lives here; below ~105 GiB the load will be killed' if ram < 105 else '-- enough for the n-gram table')

In [ ]:
import os, glob, json, time, re, torch, transformers
from transformers import AutoConfig, AutoTokenizer, Qwen4ExpForConditionalGeneration
transformers.logging.set_verbosity_error()   # the per-key load report floods the notebook; we read the same facts from `info` below
assert os.environ.get('PYTORCH_CUDA_ALLOC_CONF') == 'expandable_segments:True', 'run the install cell first (sets the allocator mode)'
# transformers 'warms up' the CUDA cache by reserving nearly the whole GPU in one block before loading.  With 222k tensors, most of
# them tiny (the 4-bit scales/zeros of 73k expert linears), the tiny ones need fresh 2 MiB segments the reserved block cannot
# provide, and the load dies at ~36% with '68 GiB reserved but unallocated'.  No flag turns the warm-up off, so we blank it.
import transformers.modeling_utils as _mu
_mu.caching_allocator_warmup = lambda *a, **k: None

cands = sorted({os.path.dirname(p) for p in glob.glob('/kaggle/input/models/**/config.json', recursive=True)})
print('model dirs found:', cands)
MODEL_DIR = cands[0]          # (set by hand if several are attached)
DATA = glob.glob('/kaggle/input/**/dataset.json', recursive=True)[0]
print('model:', MODEL_DIR); print('data:', DATA)
ram = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30
print('host RAM %.0f GiB (the n-gram table needs ~95 GiB of it)' % ram)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# device_map='auto' is wrong for this model on ONE gpu: transformers 5.16's planner reserves room for the biggest layer -- the
# layer holding the 95 GiB n-gram table -- finds it cannot fit, and sends that layer AND EVERY LAYER AFTER IT to the cpu, so the
# gpu ends up empty (the '62.7 GB of buffer for offloaded layers' warning).  So we hand it an explicit map: every module on
# cuda:0 except the n-gram table, which is left out of the map and therefore lands on the cpu, where the model's own forward
# looks it up (ngram_ids are moved to the table's device and back).  With a single-device map transformers attaches no
# offload hooks, so nothing streams the table to the gpu.
cfg = AutoConfig.from_pretrained(MODEL_DIR)
with torch.device('meta'):
    skel = Qwen4ExpForConditionalGeneration(cfg)
tables = {n for n, _ in skel.named_modules() if n.endswith('ple_embedding.ngram_embedding')}
print('n-gram tables:', sorted(tables))
dm = {}
def walk(name, mod):
    if name in tables:
        return
    if not any(t.startswith(name + '.') for t in tables):
        dm[name] = 'cuda:0'; return
    for pn, _ in list(mod.named_parameters(recurse=False)) + list(mod.named_buffers(recurse=False)):
        dm[f'{name}.{pn}'] = 'cuda:0'
    for cn, c in mod.named_children():
        walk(f'{name}.{cn}', c)
for cn, c in skel.named_children():
    walk(cn, c)
del skel
print(len(dm), 'device-map entries, e.g.', [k for k in dm if 'layers.1.' in k or k.startswith('lm')][:8])

t0 = time.time()
model, info = Qwen4ExpForConditionalGeneration.from_pretrained(MODEL_DIR, dtype='auto', device_map=dm, output_loading_info=True)
model.eval()
print('missing keys:', len(info['missing_keys']), info['missing_keys'][:3]); print('unexpected keys:', len(info['unexpected_keys']), info['unexpected_keys'][:3])
assert not any('ngram_embedding' in k for k in info['missing_keys']), 'the n-gram table was NOT loaded from the shards -- stop here and report'
devs = {}
for n, p in model.named_parameters():
    devs[str(p.device)] = devs.get(str(p.device), 0) + p.numel()
print('loaded in %.0fs' % (time.time() - t0), '| parameters by device (G):', {k: round(v / 1e9, 1) for k, v in devs.items()},
      '| gpu memory used', round(torch.cuda.memory_allocated(0) / 2**30, 1), 'GiB')
print('expected: everything on cuda:0 except ~51 G on cpu (the n-gram table); nothing on meta')
DEV = torch.device('cuda:0')

In [ ]:
def ask(prompt, max_new_tokens=500):
    msgs = [{'role': 'system', 'content': 'You answer with one JSON object and nothing else.'},
            {'role': 'user', 'content': prompt}]
    try:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tokenizer([text], return_tensors='pt').to(DEV)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)

rows = json.load(open(DATA))
print(len(rows), 'scenes; prompt chars', [len(r['prompt']) for r in rows[:5]])
t0 = time.time(); a0 = ask(rows[0]['prompt']); print('first call %.1fs' % (time.time() - t0)); print(a0[:800])

In [ ]:
answers = {}
for i, r in enumerate(rows):
    key = f"{r['game']}:{r['level']}"
    t0 = time.time()
    try:
        a = ask(r['prompt'])
    except Exception as e:
        a = f'ERROR {e!r}'
    answers[key] = {'answer': a, 'seconds': round(time.time() - t0, 1)}
    print(f"{i+1:2}/{len(rows)} {key:8} {answers[key]['seconds']:5.1f}s  {a[:120].replace(chr(10), ' ')}")
    json.dump(answers, open('/kaggle/working/reader_answers.json', 'w'), indent=1)   # (saved after every scene)
print('saved /kaggle/working/reader_answers.json')

In [ ]:
# ---- score against the gold (same logic as tools/reader_eval/score.py)
def parse_answer(text):
    m = re.search(r'\{.*\}', text or '', re.S)
    if not m: return None
    raw = m.group(0)
    for cand in (raw, re.sub(r'//[^\n]*', '', raw), re.sub(r',\s*([}\]])', r'\1', re.sub(r'//[^\n]*', '', raw))):
        try: return json.loads(cand)
        except Exception: pass
    return None

def sid(x):
    m = re.search(r'(\d+)', str(x)) if x is not None else None
    return int(m.group(1)) if m else None

n_gold = n_hit = 0
for r in rows:
    key = f"{r['game']}:{r['level']}"; a = answers.get(key, {})
    parsed = parse_answer(a.get('answer', ''))
    col = {t['sid']: t['colour'] for t in r['things']}
    reads = [(rd.get('kind'), col.get(sid(rd.get('mover'))), col.get(sid(rd.get('target'))), (rd.get('why') or '')[:60])
             for rd in ((parsed or {}).get('readings') or [])[:4]]
    gold = r.get('gold'); hit = None
    if gold:
        n_gold += 1
        hit = any(mc in gold['mover'] and tc in gold['target'] and (not gold.get('kind') or kd in gold['kind']) for kd, mc, tc, _ in reads[:2])
        n_hit += hit
    mark = '  ' if hit is None else ('OK' if hit else '--')
    print(f"{mark} {key:8} {a.get('seconds', 0):5.1f}s {[(k, m, t) for k, m, t, _ in reads]}  laws={(parsed or {}).get('laws')}")
    if gold: print('          gold:', gold['reading'])
print(f'\n{n_hit}/{n_gold} gold scenes read as a person reads them')